In [24]:
import json
import re

from typing import List
from pydantic import BaseModel

LETTER_BOUNDARY = re.compile(
    r"(^#{1,2} .+|^\*{3}$|^-{3}$)",
    re.MULTILINE,
)

CATCHWORD = re.compile(r"^[A-Z]{1,3}\s*[\d\[\]]*$")


def extract_book_page(header_line: str) -> str | None:
    m = re.search(r"\b(\d+)\b", header_line)
    return m.group(1) if m else None


def clean_title(s: str) -> str:
    s = s.replace("#", "")
    s = s.replace("*", "")
    s = s.replace("-", "")
    return s.strip()


def is_all_caps_title(line: str) -> bool:
    clean = re.sub(r"[#*\s]+", "", line)
    if len(clean) < 6:
        return False
    if CATCHWORD.match(line.strip()):
        return False
    letters = [c for c in line if c.isalpha()]
    if not letters:
        return False
    return sum(1 for c in letters if c.isupper()) / len(letters) > 0.75


with open("./data/ocr.json", "r") as f:
    doc = json.load(f)
    doc = doc["pages"]


class LettersModel(BaseModel):
    pdf_pages: List[int]
    markdown: str


letters: List[LettersModel] = []
current: LettersModel | None = None

for page in doc:
    pdf_page: int = page["index"]
    md: str = page["markdown"]
    body = md.split("\n\n")[1:]
    if pdf_page < 46 or pdf_page >= 664:
        continue
    pattern = r"([A-ZÀ-Ö.']{1,}\s[A-ZÀ-Ö.']{2,})"
    # pattern = r"([A-Z.']*\s[A-Z.']{2,})"

    # lines = re.split(r"(---|\*\*\*)(\s*)?\n*", md)
    lines = md.split("\n\n")

    for line in lines:
        if line.find("LETTRES ET MEMOIRES") > -1:
            continue
        CAPS = re.findall(pattern, line)
        if CAPS:
            start_opts = (
                CAPS[0],
                f"#{CAPS[0]}",
                f"# {CAPS[0]}",
                f"##{CAPS[0]}",
                f"## {CAPS[0]}",
                f"### {CAPS[0]}",
                f"###{CAPS[0]}",
                f"** {CAPS[0]}",
                f"**{CAPS[0]}",
            )
            if line.startswith(start_opts) and re.search(
                r"\b(A|AV|AVX|AU|AUX|DU|DE|DES|D')\b", line
            ):
                if current:
                    letters.append(current)
                current = LettersModel(pdf_pages=[pdf_page], markdown=line)
            else:
                if current:
                    current.markdown += "\n\n" + line
                    if not pdf_page in current.pdf_pages:
                        current.pdf_pages.append(pdf_page)
        else:
            if current:
                current.markdown += "\n\n" + line
                if not pdf_page in current.pdf_pages:
                    current.pdf_pages.append(pdf_page)

print(len(letters))
letters_to_remove = []
new_letters = []
for i in range(len(letters) - 1):
    md = letters[i].markdown
    if len(md) < 100:
        letters[i + 1].markdown = md + " " + letters[i + 1].markdown
        letters_to_remove.append(i)
for idx, l in enumerate(letters):
    if idx in letters_to_remove:
        continue
    new_letters.append(l)
letters = new_letters
print(letters_to_remove)
print(len(letters))

for s in letters:
    print(s)

286
[9, 12, 64, 66, 158, 170, 175, 179, 196, 199, 201, 209, 212, 215, 218, 222, 226, 231, 279, 282]
266
pdf_pages=[46] markdown="LE PAPE PAVL III. AV ROT FRANCOIS I.\n\nCe Bref porte creance à l'Euesgue de Verone, et comme l'on verra par autres Lettres, c'estoit pour le Concile et pour la Paix.\n\nPAVLVS PAPA III.\n\nCHARISSIME in Christo Fili noster salutem &amp; Apostolicaem Benedictionem mandanimus venerabili Fratri Ioanni Matthæo Episcopo Veronenfi, quem cum Cardinale Angliae Legato nostro minimus ve Majestati tux quædam ex parte nostra teserret, hortamur illam in Domino ve ipsius Ioannis Matthei Episcopi verbis plenam fidem perinde habere velit, ac fi nos ipsi præsentes cum ea colloquæ remus. Datum Romæ apud Sanctum Petrum sub Annulo Piscatoris die 17. Februarij 1337. Pontificatus nostri anno tertio.\n\nPlus bas il y a d'une autre main, comme du Pape nostre,\n\nDPISCOPVM hunc Veronenfem ob eximias virtutes suas amandamus &amp; in his omnibus quæ ad Christianam pietarem pertinent f

In [29]:
import polars as pl

df = pl.DataFrame(
    {
        "markdown": [l.markdown for l in letters],
        "pdf_pages": [l.pdf_pages for l in letters],
        "title": [clean_title(l.markdown.split("\n")[0]) for l in letters],
    }
)
df = df.with_row_index("id", offset=1000)

df.write_parquet("./data/ribier_v1.parquet")

In [30]:
df1 = pl.read_parquet("./data/ribier_v1.parquet")
df1

id,markdown,pdf_pages,title
u32,str,list[i64],str
1000,"""LE PAPE PAVL III. AV ROT FRANC…",[46],"""LE PAPE PAVL III. AV ROT FRANC…"
1001,"""LETTRE DE MONLVC AV CARDINAL D…","[46, 47]","""LETTRE DE MONLVC AV CARDINAL D…"
1002,"""LETTRE DES AVOTERS ET CONSEILL…","[47, 48]","""LETTRE DES AVOTERS ET CONSEILL…"
1003,"""REMARQUES SVR LADITE VILLE DE …","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …"
1004,"""## LE GRAND M. DE FRANCE MONTM…","[54, 55]","""LE GRAND M. DE FRANCE MONTMORE…"
…,…,…,…
1261,"""DE L'AUBESPINE. --- ## LE ROY…","[656, 657]","""DE L'AUBESPINE."""
1262,"""# A MONSIEUR LE DAVHIN. Le Ca…","[657, 658]","""A MONSIEUR LE DAVHIN."""
1263,"""LE CARDINAL DE BOLOGNE ## AV R…","[658, 659]","""LE CARDINAL DE BOLOGNE AV ROT…"
